In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('vaccination.db')

# Quick verify — tables sahi se load hain
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables:", cursor.fetchall())

Tables: [('coverage',), ('incidence',), ('cases',), ('coverage_groups',), ('incidence_groups',), ('cases_groups',), ('vaccine_introduction',), ('vaccine_schedule',)]


In [3]:
query1 = """
SELECT NAME, COVERAGE
FROM coverage
WHERE COVERAGE_CATEGORY = 'WUENIC' AND YEAR = 2023 AND ANTIGEN = 'DTPCV3'
ORDER BY COVERAGE DESC
LIMIT 10
"""
print("Top 10 countries by DTP3 coverage (2023):")
print(pd.read_sql(query1, conn))

query2 = """
SELECT c.NAME, c.YEAR, c.COVERAGE, i.INCIDENCE_RATE
FROM coverage c
JOIN incidence i ON c.CODE = i.CODE AND c.YEAR = i.YEAR
WHERE c.ANTIGEN = 'MCV1' AND c.COVERAGE_CATEGORY = 'WUENIC'
  AND c.COVERAGE > 90 AND i.DISEASE = 'Measles' AND i.INCIDENCE_RATE > 10
ORDER BY i.INCIDENCE_RATE DESC
LIMIT 15
"""
print("\nCountries with high MCV1 coverage (>90%) but still notable measles incidence:")
print(pd.read_sql(query2, conn))

Top 10 countries by DTP3 coverage (2023):
                         NAME  COVERAGE
0                     Bahrain      99.0
1           Brunei Darussalam      99.0
2                      Bhutan      99.0
3                  Costa Rica      99.0
4                        Cuba      99.0
5                        Fiji      99.0
6                      Greece      99.0
7                     Hungary      99.0
8  Iran (Islamic Republic of)      99.0
9                  Kazakhstan      99.0

Countries with high MCV1 coverage (>90%) but still notable measles incidence:
Empty DataFrame
Columns: [NAME, YEAR, COVERAGE, INCIDENCE_RATE]
Index: []


In [4]:
# SQL-based validation: high MCV1 coverage with notable measles incidence

validation_query = """
SELECT 
    c.NAME,
    c.YEAR,
    c.COVERAGE,
    i.INCIDENCE_RATE
FROM coverage c
JOIN incidence i
    ON c.CODE = i.CODE
    AND c.YEAR = i.YEAR
WHERE c.ANTIGEN = 'MCV1'
  AND c.COVERAGE_CATEGORY = 'WUENIC'
  AND c.COVERAGE > 90
  AND i.DISEASE = 'MEASLES'
  AND i.INCIDENCE_RATE > 10
ORDER BY i.INCIDENCE_RATE DESC
LIMIT 15;
"""

validation_result = pd.read_sql(validation_query, conn)

print("High MCV1 coverage (>90%) with notable measles incidence:")
print(validation_result)

High MCV1 coverage (>90%) with notable measles incidence:
               NAME    YEAR  COVERAGE  INCIDENCE_RATE
0              Niue  1991.0      99.0         49347.5
1   Solomon Islands  1989.0      92.0         42887.6
2           Albania  1989.0      96.0         41723.4
3      Cook Islands  1989.0      99.0         40810.3
4             Samoa  2019.0      96.0         27109.4
5          Maldives  1995.0      96.0         11961.9
6             Tonga  1997.0      97.0         10737.6
7          Mongolia  2016.0      98.0          9955.6
8            Malawi  2010.0      93.0          8006.8
9        Seychelles  1998.0      93.0          7754.9
10         Mongolia  2015.0      98.0          6839.9
11            Tonga  2019.0      99.0          6236.4
12            Nauru  1997.0      99.0          5687.9
13           Tuvalu  1996.0      94.0          5459.2
14           Monaco  1992.0      98.0          5180.2


In [5]:
# Diagnostic - DISEASE column mein exact values kya hain
cursor.execute("SELECT DISTINCT DISEASE FROM incidence LIMIT 20")
print("DISEASE values:", cursor.fetchall())

# Diagnostic - ANTIGEN column mein exact values
cursor.execute("SELECT DISTINCT ANTIGEN FROM coverage WHERE ANTIGEN LIKE '%MCV%'")
print("MCV antigen values:", cursor.fetchall())

# Simple counts, bina join ke, step by step
cursor.execute("SELECT COUNT(*) FROM coverage WHERE ANTIGEN='MCV1' AND COVERAGE_CATEGORY='WUENIC' AND COVERAGE>90")
print("Coverage>90 rows (MCV1, WUENIC):", cursor.fetchone())

cursor.execute("SELECT COUNT(*) FROM incidence WHERE DISEASE='Measles' AND INCIDENCE_RATE>10")
print("Measles incidence>10 rows:", cursor.fetchone())

DISEASE values: [('CRS',), ('DIPHTHERIA',), ('INVASIVE_MENING',), ('MEASLES',), ('MUMPS',), ('NTETANUS',), ('PERTUSSIS',), ('POLIO',), ('RUBELLA',), ('TTETANUS',), ('TYPHOID',), ('YFEVER',), ('JAPENC',)]
MCV antigen values: [('MCV1',), ('MCV2',)]
Coverage>90 rows (MCV1, WUENIC): (3288,)
Measles incidence>10 rows: (0,)


In [6]:
query2 = """
SELECT c.NAME, c.YEAR, c.COVERAGE, i.INCIDENCE_RATE
FROM coverage c
JOIN incidence i ON c.CODE = i.CODE AND c.YEAR = i.YEAR
WHERE c.ANTIGEN = 'MCV1' AND c.COVERAGE_CATEGORY = 'WUENIC'
  AND c.COVERAGE > 90 AND i.DISEASE = 'MEASLES' AND i.INCIDENCE_RATE > 10
ORDER BY i.INCIDENCE_RATE DESC
LIMIT 15
"""
print("Countries with high MCV1 coverage (>90%) but still notable measles incidence:")
print(pd.read_sql(query2, conn))

Countries with high MCV1 coverage (>90%) but still notable measles incidence:
               NAME    YEAR  COVERAGE  INCIDENCE_RATE
0              Niue  1991.0      99.0         49347.5
1   Solomon Islands  1989.0      92.0         42887.6
2           Albania  1989.0      96.0         41723.4
3      Cook Islands  1989.0      99.0         40810.3
4             Samoa  2019.0      96.0         27109.4
5          Maldives  1995.0      96.0         11961.9
6             Tonga  1997.0      97.0         10737.6
7          Mongolia  2016.0      98.0          9955.6
8            Malawi  2010.0      93.0          8006.8
9        Seychelles  1998.0      93.0          7754.9
10         Mongolia  2015.0      98.0          6839.9
11            Tonga  2019.0      99.0          6236.4
12            Nauru  1997.0      99.0          5687.9
13           Tuvalu  1996.0      94.0          5459.2
14           Monaco  1992.0      98.0          5180.2


In [7]:
cursor.execute("SELECT DISTINCT DENOMINATOR FROM incidence")
print("Denominator values:", cursor.fetchall())

Denominator values: [('per 10,000 live births',), ('per 1,000,000 total population',), ('per 1,000 live births',), ('per 1,000,000 <15 population',)]


In [8]:
cursor.execute("SELECT DISTINCT DENOMINATOR FROM incidence WHERE DISEASE='MEASLES'")
print("Measles-specific denominator:", cursor.fetchall())

Measles-specific denominator: [('per 1,000,000 total population',)]


In [9]:
# Query 3: Jo vaccines sabse der se globally introduce hue (average introduction year sabse zyada)
query3 = """
SELECT DESCRIPTION, COUNT(*) as countries_introduced, AVG(YEAR) as avg_intro_year
FROM vaccine_introduction
WHERE INTRO = 'Yes'
GROUP BY DESCRIPTION
ORDER BY avg_intro_year DESC
LIMIT 10
"""
print("Vaccines with latest average global introduction:")
print(pd.read_sql(query3, conn))

Vaccines with latest average global introduction:
                                       DESCRIPTION  countries_introduced  \
0              HPV (Human Papilloma Virus) vaccine                  1144   
1                                Rotavirus vaccine                  1200   
2             PCV (Pneumococcal conjugate vaccine)                  1828   
3                                Varicella vaccine                   514   
4                              Hepatitis A vaccine                   289   
5                            Japanese Encephalitis                   130   
6  Meningococcal meningitis vaccines (all strains)                   644   
7         IPV (Inactivated polio vaccine) 2nd dose                  1523   
8                                          Typhoid                    30   
9                 aP (acellular pertussis) vaccine                  1313   

   avg_intro_year  
0     2017.724650  
1     2017.545000  
2     2016.845186  
3     2016.375486  
4     2016.13